In [9]:
!pip install tabula-py pandas sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.2 MB/s eta 0:00:00


In [10]:
import tabula
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [11]:
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
if GEMINI_API_KEY is None:
  raise ValueError("No GEMINI_API_KEY found in Colab Secrets")

  genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
path = "/content/order.pdf"
tables = tabula.read_pdf(path,pages="all")
df = pd.concat(tables,ignore_index=True)
print(df)

In [ ]:
meta_data = df.describe(include= "all").fillna("")
meta_data

,order_id,order_status,customer,order_date,order_quantity,sales
count,20.0,20,20,20,20.0,20.0
unique,,2,16,19,,
top,,Order\rFinished,Carlos Soltero,4/8/2012,,
freq,,19,3,2,,
mean,1003.65,,,,27.3,4032125.35
std,514.490272,,,,13.010522,6859259.145341
min,3.0,,,,6.0,118060.0
25%,635.5,,,,15.75,342045.0
50%,964.0,,,,26.5,764750.0
75%,1443.75,,,,35.75,4114145.0


In [ ]:
print(df.describe())

          order_id  order_quantity         sales
count    20.000000       20.000000  2.000000e+01
mean   1003.650000       27.300000  4.032125e+06
std     514.490272       13.010522  6.859259e+06
min       3.000000        6.000000  1.180600e+05
25%     635.500000       15.750000  3.420450e+05
50%     964.000000       26.500000  7.647500e+05
75%    1443.750000       35.750000  4.114145e+06
max    1792.000000       49.000000  2.405646e+07


In [13]:
documents1 = []
for _, row in df.iterrows():
  doc = (
      f"Order ID: {row['order_id']} has status {row['order_status']}\n "
      f"Customer is {row['customer']}."
      f"Order date is {row['order_date']}."
      f"Quantity Ordered is {row['order_quantity']}."
      f"Total Sale Amount is {row['sales']}."
  )
  documents1.append(doc)
print(documents1)

NameError: name 'df' is not defined

In [ ]:
for _, row in df.iterrows():
  print(row)

order_id                             3
order_status           Order\rFinished
customer          Muhammed Mac\rIntyre
order_date                  13/10/2010
order_quantity                       6
sales                           523080
Name: 0, dtype: object
order_id                      293
order_status      Order\rFinished
customer             Barry French
order_date              1/10/2012
order_quantity                 49
sales                    20246040
Name: 1, dtype: object
order_id                      483
order_status      Order\rFinished
customer            Clay Rozendal
order_date              10/7/2011
order_quantity                 30
sales                     9931519
Name: 2, dtype: object
order_id                      515
order_status      Order\rFinished
customer           Carlos Soltero
order_date              28/8/2010
order_quantity                 19
sales                      788540
Name: 3, dtype: object
order_id                      613
order_status      Order\rFin

In [ ]:
documents = []
for _, row in df.iterrows():
  doc = (
      f"Order ID: {row['order_id']} has status {row['order_status']}\n "
      f"Customer is {row['customer']}."
      f"Order date is {row['order_date']}."
      f"Quantity Ordered is {row['order_quantity']}."
      f"Total Sale Amount is {row['sales']}."
  )
  documents.append(doc)
  print(documents)

['Order ID: 3 has status Order\rFinished\n Customer is Muhammed Mac\rIntyre.Order date is 13/10/2010.Quantity Ordered is 6.Total Sale Amount is 523080.']
['Order ID: 3 has status Order\rFinished\n Customer is Muhammed Mac\rIntyre.Order date is 13/10/2010.Quantity Ordered is 6.Total Sale Amount is 523080.', 'Order ID: 293 has status Order\rFinished\n Customer is Barry French.Order date is 1/10/2012.Quantity Ordered is 49.Total Sale Amount is 20246040.']
['Order ID: 3 has status Order\rFinished\n Customer is Muhammed Mac\rIntyre.Order date is 13/10/2010.Quantity Ordered is 6.Total Sale Amount is 523080.', 'Order ID: 293 has status Order\rFinished\n Customer is Barry French.Order date is 1/10/2012.Quantity Ordered is 49.Total Sale Amount is 20246040.', 'Order ID: 483 has status Order\rFinished\n Customer is Clay Rozendal.Order date is 10/7/2011.Quantity Ordered is 30.Total Sale Amount is 9931519.']
['Order ID: 3 has status Order\rFinished\n Customer is Muhammed Mac\rIntyre.Order date is 1

In [14]:
embedding_modal = SentenceTransformer('all-MiniLM-L6-v2')
embedding = embedding_modal.encode(documents1, convert_to_numpy=True,show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches: 0it [00:00, ?it/s]

In [ ]:
print(embedding)

[[-0.03465751 -0.04500803  0.05368118 ... -0.09129117 -0.05807808
   0.05143303]
 [-0.0245116  -0.02223356 -0.00104521 ... -0.05593821 -0.06410709
  -0.01400049]
 [-0.05636995 -0.03436095  0.01603419 ... -0.03779768 -0.08158956
  -0.01815924]
 ...
 [ 0.0262889  -0.00157958  0.07718559 ... -0.04049924 -0.02472917
  -0.02574381]
 [-0.08880605  0.01342723  0.04455399 ... -0.04211368 -0.01552752
   0.00321055]
 [-0.06313058 -0.00841524  0.01223989 ... -0.01195232 -0.0963881
   0.00539362]]


In [15]:
dim = embedding.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embedding)
faiss.write_index(index,"faiss_index.faiss")

IndexError: tuple index out of range

In [16]:
def retrive_context(query,k=3):
  query_embedding = embedding_modal.encode(query)
  distances,indices = index.search(query_embedding,k)
  return "\n".join ([documents1[i]for i in indices[0]])

In [ ]:
genratation_configur = {
    'trmp': 0.4,
    'max_output_tokens': 512
}
print(genratation_configur)
gemini_model = genai.GenerativeModel(model_name='model/gemini-2.5-flash',generation_config=genratation_configur)

{'trmp': 0.4, 'max_output_tokens': 512}


In [17]:
chat_history = []
def chat_with_bot(user_input):
  global chat_history

  context = retrive_context(user_input)
  # Renamed 'prompt' to 'model_query_prompt' to avoid conflict and for clarity
  model_query_prompt = f"""
  you are a helpful conversational data analyst assistant.Plesae refer the context below and answer the user's question.
  context:
 {context}
 user_question:
 {user_input}
 Rules:
 - Be Conconversational
 - Answer only using context
 - if you dont know the answer say you don't know
"""
  response = gemini_model.generate_content(model_query_prompt)
  answer = response.text
  chat_history.append({"user":user_input, "bot" :answer})
  return answer

In [ ]:
print("order analytics chat Boat Ready !!!")
print("type'exir' to stop\n")

while True:
  user_input = input("user: ")
  if user_input.lower() in ['exit','quit','bye']:
    print("Good Bye !!!")
    break
  response = chat_with_bot(user_input)
  print(f"Bot :{response}")
  print("-"*60)

order analytics chat Boat Ready !!!
type'exir' to stop



In [7]:
def retrive_context(query,k=3):
  query_embedding = embedding_modal.encode(query)
  distances,indices = index.search(query_embedding,k)
  return "\n".join ([documents1[i]for i in indices[0]])